# Practical Lab — Cardiac Patient Digital Twin

**Part 2** of *Knowledge Graphs as Semantic Bridges between CPS/IoT and Patient Digital Twins in Healthcare*.

Build a working KG that integrates **wearable ECG + clinical history + clinical events** and query it with **SPARQL** to answer real cardiology questions.

Open `README.md` beside this notebook for the full step-by-step guide. Run cells top-to-bottom.

## Phase 1 — Setup & data exploration (10 min)

In [ ]:
# Environment verification — must print all versions with no error
import sys
print("Python:", sys.version.split()[0])
import rdflib, pandas, networkx, matplotlib, IPython
for m in (rdflib, pandas, networkx, matplotlib, IPython):
    print(f"{m.__name__:12s} {m.__version__}")

In [ ]:
import pandas as pd
import json

history = pd.read_csv("clinical_history.csv")
events  = pd.read_csv("clinical_events.csv")
with open("ecg_data.json") as f:
    ecg = json.load(f)

print("history:", history.shape, "| events:", events.shape, "| ecg:", len(ecg), "records")
history.head()

In [ ]:
# Exploratory queries with Pandas
history["n_dx"] = history["diagnoses"].str.split(";").str.len()
print("Patients per diagnosis code (exploded):")
history.assign(dx=history["diagnoses"].str.split(";")) \
       .explode("dx")["dx"].value_counts()

## Phase 2 — Build the knowledge graph (25 min)

In [ ]:
from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS, XSD

CARDIO = Namespace("http://example.org/cardio#")
SNOMED = Namespace("http://snomed.info/id/")
LOINC  = Namespace("http://loinc.org/")
FHIR   = Namespace("http://hl7.org/fhir/")

g = Graph()
g.bind("cardio", CARDIO)
g.bind("snomed", SNOMED)
g.bind("loinc", LOINC)
g.bind("fhir", FHIR)
g.parse("healthcare_cardio.ttl", format="turtle")   # TBox + code/interaction instances

def pt(pid): return CARDIO[f"Patient/{pid}"]

# (1) History -> patients, diagnoses, drugs
for _, r in history.iterrows():
    p = pt(r.patient_id)
    g.add((p, RDF.type, CARDIO.Patient))
    g.add((p, CARDIO.hasName, Literal(r["name"])))
    g.add((p, CARDIO.hasAge, Literal(int(r.age), datatype=XSD.integer)))
    g.add((p, CARDIO.hasSex, Literal(r.sex)))
    for code in str(r.diagnoses).split(";"):
        if code.strip():
            g.add((p, CARDIO.hasDiagnosis, URIRef(SNOMED[code.strip()])))
    for drug in str(r.medications).split(";"):
        if drug.strip():
            g.add((p, CARDIO.takes, URIRef(CARDIO[drug.strip()])))

# (2) Wearable ECG -> sensor + measurements
for rec in ecg:
    p = pt(rec["patient_id"])
    sensor = URIRef(CARDIO[f"Sensor/{rec['patient_id']}-ECG"])
    g.add((p, CARDIO.monitoredBy, sensor))
    g.add((sensor, RDF.type, CARDIO.WearableSensor))
    obs = URIRef(CARDIO[f"ECG/{rec['record_id']}"])
    g.add((sensor, CARDIO.produced, obs))
    g.add((obs, RDF.type, CARDIO.ECGMeasurement))
    g.add((obs, CARDIO.hasTimestamp, Literal(rec["timestamp"], datatype=XSD.dateTime)))
    g.add((obs, CARDIO.hasHeartRate, Literal(int(rec["heart_rate"]), datatype=XSD.integer)))
    g.add((obs, CARDIO.hasQTDuration, Literal(int(rec["qt_ms"]), datatype=XSD.integer)))
    g.add((obs, CARDIO.hasQRS, Literal(int(rec["qrs_ms"]), datatype=XSD.integer)))
    g.add((obs, CARDIO.hasRhythm, Literal(rec["rhythm"])))

# (3) Clinical events
for _, r in events.iterrows():
    p = pt(r.patient_id)
    e = URIRef(CARDIO[f"Event/{r.event_id}"])
    g.add((p, CARDIO.hasEvent, e))
    g.add((e, RDF.type, CARDIO.ClinicalEvent))
    g.add((e, CARDIO.hasTimestamp, Literal(f"{r.event_date}T00:00:00", datatype=XSD.dateTime)))
    g.add((e, CARDIO.hasEventType, Literal(r.event_type)))
    if pd.notna(r.concept_code) and str(r.concept_code).strip():
        g.add((e, CARDIO.hasConceptCode, Literal(str(r.concept_code).strip())))
    if pd.notna(r.value):
        g.add((e, CARDIO.hasValue, Literal(float(r.value), datatype=XSD.double)))
    if pd.notna(r.severity) and str(r.severity).strip():
        g.add((e, CARDIO.hasSeverity, Literal(r.severity)))

print("Triples in KG:", len(g))
g.serialize(destination="cardiac_kg.ttl", format="turtle")

In [ ]:
# Validation - verify the KG was built correctly
assert len(g) > 600, f"Expected >600 triples, got {len(g)}"
assert len(list(g.query("SELECT ?p WHERE { ?p a cardio:Patient }"))) == 20, "Expected 20 patients"
assert len(list(g.query("SELECT ?e WHERE { ?e a cardio:ECGMeasurement }"))) == 100, "Expected 100 ECG measurements"
assert len(list(g.query("SELECT ?e WHERE { ?e a cardio:ClinicalEvent }"))) == 50, "Expected 50 clinical events"
print("✓ KG validation passed:", len(g), "triples, 20 patients, 100 ECG readings, 50 events")

In [ ]:
# Serialise & visualise an ego subgraph (one patient, radius 2)
import networkx as nx
from rdflib.extras.external_graph_libs import rdflib_to_networkx_multidigraph
import matplotlib.pyplot as plt
%matplotlib inline

nxg = rdflib_to_networkx_multidigraph(g)
center = URIRef(CARDIO["Patient/P009"])
ego = nx.ego_graph(nxg, center, radius=2)
pos = nx.spring_layout(ego, seed=7)
nx.draw(ego, pos, node_size=350, node_color="#9ecae1", edge_color="#9aa", with_labels=False)
nx.draw_networkx_labels(ego, pos, {n: str(n).split("/")[-1] for n in ego.nodes}, font_size=7)
plt.title("Patient P009 - ego graph (radius 2)")
plt.axis("off"); plt.show()

## Phase 3 — SPARQL queries & clinical inference (20 min)

In [ ]:
# Q1 (simple) - all patients with atrial fibrillation (SNOMED 49436004)
q1 = """
PREFIX cardio: <http://example.org/cardio#>
PREFIX snomed: <http://snomed.info/id/>
SELECT ?patient ?name WHERE {
  ?patient a cardio:Patient ;
           cardio:hasName ?name ;
           cardio:hasDiagnosis snomed:49436004 .
}
"""
print("Patients with atrial fibrillation:")
for row in g.query(q1):
    print(" ", row.patient.split("/")[-1], "-", row.name)

In [ ]:
# Q2 (filter) - tachycardia readings (HR > 100), ordered
q2 = """
PREFIX cardio: <http://example.org/cardio#>
SELECT ?patient ?hr ?ts WHERE {
  ?patient cardio:monitoredBy ?sensor .
  ?sensor cardio:produced ?obs .
  ?obs cardio:hasHeartRate ?hr ; cardio:hasTimestamp ?ts .
  FILTER(?hr > 100)
}
ORDER BY DESC(?hr)
"""
rows = list(g.query(q2))
print(f"Tachycardia readings (HR>100): {len(rows)}")
for row in rows[:8]:
    print(" ", row.patient.split("/")[-1], row.hr, str(row.ts))

In [ ]:
# Q3 (pattern match) - QT prolongation (>470 ms), a pro-arrhythmia flag
q3 = """
PREFIX cardio: <http://example.org/cardio#>
SELECT DISTINCT ?patient ?qt WHERE {
  ?patient cardio:monitoredBy ?s .
  ?s cardio:produced ?obs .
  ?obs cardio:hasQTDuration ?qt .
  FILTER(?qt > 470)
}
ORDER BY DESC(?qt)
"""
print("QT prolongation (>470 ms):")
for row in g.query(q3):
    print(" ", row.patient.split("/")[-1], row.qt, "ms")

In [ ]:
# Q4 (complex, multi-source) - AF patients taking two drugs that interact
q4 = """
PREFIX cardio: <http://example.org/cardio#>
PREFIX snomed: <http://snomed.info/id/>
SELECT DISTINCT ?patient ?d1 ?d2 WHERE {
  ?patient cardio:hasDiagnosis snomed:49436004 ;
           cardio:takes ?d1 ; cardio:takes ?d2 .
  ?d1 (^cardio:interactsWith|cardio:interactsWith) ?d2 .
  FILTER(?d1 != ?d2 && str(?d1) < str(?d2))
}
"""
print("AF patients on interacting drug pairs:")
for row in g.query(q4):
    print(" ", row.patient.split("/")[-1], "->", row.d1.split("#")[-1], "+", row.d2.split("#")[-1])

## Phase 4 — Guided independent exercises (25 min)

**Ex 1 (Medium)** — Extend the ontology with a new drug + interaction (starter below).
**Ex 2 (Advanced)** — Multi-criteria at-risk query (AF + interacting pair + tachycardia). See README §8.
**Ex 3 (Open)** — Architectural improvement proposal (3 slides). See README §8.

In [ ]:
# Exercise 1 starter - complete the TODOs, then query all interactions to verify the new pair (Q4 filters on AF, and P016 is not AF).
Sotalol = URIRef(CARDIO["Sotalol"])
# TODO: type it as a Drug and label it
# g.add((Sotalol, RDF.type, CARDIO.Drug))
# g.add((Sotalol, RDFS.label, Literal("Sotalol")))
# TODO: assert it interactsWith Amiodarone (additive QT risk)
# g.add((Sotalol, CARDIO.interactsWith, CARDIO["Amiodarone"]))
# TODO: assign it to Patient/P016
# g.add((URIRef(CARDIO["Patient/P016"]), CARDIO.takes, Sotalol))
# After adding the triples, confirm the new Sotalol + Amiodarone pair with an interactions query (see README Sec. 8).

## Phase 5 — Discussion & wrap-up (10 min)

See README §9 (discussion prompts) and §10 (self-assessment). Key questions:

1. Is this KG *clinically* valid? What missing element would make a Q4 inference unsafe?
2. `interactsWith` is symmetric. Which clinical relationship would be dangerous to model as symmetric?
3. What would have to change in the **schema** (not the query) to answer *"patients who became at-risk in the last 24h"*?

---
*Solutions and assessment criteria for lecturers are in `README.md` §8.*